In [12]:
!pip install gensim

In [25]:
from google.colab import files
files.upload()

Saving sentiment_model.pth to sentiment_model (4).pth


{'sentiment_model (4).pth': b'PK\x03\x04\x00\x00\x08\x08\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x18\x00\n\x00sentiment_model/data.pklFB\x06\x00ZZZZZZ\x80\x02ccollections\nOrderedDict\nq\x00)Rq\x01(X\n\x00\x00\x00fc1.weightq\x02ctorch._utils\n_rebuild_tensor_v2\nq\x03((X\x07\x00\x00\x00storageq\x04ctorch\nFloatStorage\nq\x05X\x01\x00\x00\x000q\x06X\x03\x00\x00\x00cpuq\x07M\x00\x96tq\x08QK\x00K\x80M,\x01\x86q\tM,\x01K\x01\x86q\n\x89h\x00)Rq\x0btq\x0cRq\rX\x08\x00\x00\x00fc1.biasq\x0eh\x03((h\x04h\x05X\x01\x00\x00\x001q\x0fh\x07K\x80tq\x10QK\x00K\x80\x85q\x11K\x01\x85q\x12\x89h\x00)Rq\x13tq\x14Rq\x15X\n\x00\x00\x00fc2.weightq\x16h\x03((h\x04h\x05X\x01\x00\x00\x002q\x17h\x07K\x80tq\x18QK\x00K\x01K\x80\x86q\x19K\x80K\x01\x86q\x1a\x89h\x00)Rq\x1btq\x1cRq\x1dX\x08\x00\x00\x00fc2.biasq\x1eh\x03((h\x04h\x05X\x01\x00\x00\x003q\x1fh\x07K\x01tq QK\x00K\x01\x85q!K\x01\x85q"\x89h\x00)Rq#tq$Rq%u}q&X\t\x00\x00\x00_metadataq\'h\x00)Rq((X\x00\x00\x00\x00q)}q*X\x07\x00\x0

In [26]:
import re
import numpy as np
import torch
import torch.nn as nn
import gensim.downloader as api

In [27]:
word2vec_model = api.load("word2vec-google-news-300")

In [28]:
def sentence_vector(text):
    words = text.split()
    vectors = []

    for w in words:
        if w in word2vec_model:
            vectors.append(word2vec_model[w])

    if len(vectors) == 0:
        return np.zeros(300)

    return np.mean(vectors, axis=0)

In [29]:
class SentimentClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(300, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.sigmoid(x)
        return x

In [30]:
model = SentimentClassifier()
model.load_state_dict(torch.load("sentiment_model.pth", map_location="cpu"))
model.eval()

SentimentClassifier(
  (fc1): Linear(in_features=300, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [31]:
def predict_sentiment(review):
    cleaned = review.lower()

    vector = sentence_vector(cleaned)
    vector = torch.tensor(vector, dtype=torch.float32).unsqueeze(0)

    with torch.no_grad():
        output = model(vector).item()

    sentiment = "Positive" if output >= 0.5 else "Negative"

    print("Review:", review)
    print("Prediction:", sentiment)
    print("Confidence:", output)

In [32]:
predict_sentiment("This movie was amazing!")

Review: This movie was amazing!
Prediction: Negative
Confidence: 0.006640600040555


In [34]:
predict_sentiment("can you help me")

Review: can you help me
Prediction: Positive
Confidence: 0.9908652901649475


In [35]:
predict_sentiment("i am happy")

Review: i am happy
Prediction: Positive
Confidence: 0.9998871088027954
